# Pruning Basics

This notebook removes small weights from a model after training.

Goals:
- train a small MLP
- prune a fraction of its weights
- measure sparsity and accuracy


In [ ]:
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.utils.prune as prune
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)

digits = load_digits()
X = torch.tensor(digits.data / 16.0, dtype=torch.float32)
y = torch.tensor(digits.target, dtype=torch.long)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=256)


In [ ]:
class PruneMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(64, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 10)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)


def evaluate(model):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for xb, yb in test_loader:
            preds = model(xb).argmax(dim=1)
            correct += (preds == yb).sum().item()
            total += len(yb)
    return correct / total


def sparsity(layer):
    zeros = torch.sum(layer.weight == 0).item()
    total = layer.weight.nelement()
    return zeros / total


model = PruneMLP()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(8):
    model.train()
    for xb, yb in train_loader:
        loss = loss_fn(model(xb), yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

before_acc = evaluate(model)


In [ ]:
parameters_to_prune = (
    (model.fc1, "weight"),
    (model.fc2, "weight"),
)

prune.global_unstructured(
    parameters_to_prune,
    pruning_method=prune.L1Unstructured,
    amount=0.4,
)

after_acc = evaluate(model)

pd.DataFrame(
    [
        {
            "metric": "accuracy_before",
            "value": before_acc,
        },
        {
            "metric": "accuracy_after",
            "value": after_acc,
        },
        {
            "metric": "fc1_sparsity",
            "value": sparsity(model.fc1),
        },
        {
            "metric": "fc2_sparsity",
            "value": sparsity(model.fc2),
        },
    ]
)
